## Problem Statement

As a developer, you are tasked with building an AI-powered multi-agent
system to answer domain-specific questions using both static and dynamic
sources. The system comprises a **Router Agent**, which decides the
retrieval path based on the question, and a **Retriever Agent**, which
executes the retrieval from the chosen source and formulates the answer,
followed by an **Answer Generation Agent** that produces the final
source-grounded response.

Available tools include a PDF search tool (`crewai_tools.PDFSearchTool`)
for static, domain-specific content (here, the "Attention Is All You Need"
paper) and a web search tool (`TavilySearchResults`) for retrieving fresh
information from the internet. An optional direct generation path uses the
LLM without retrieval for general questions.

The **Router Agent** classifies each question between PDF search, web
search, or a direct answer, and the **Retriever Agent** is built for that
question with only the matching tool bound to it, so it can only search the
source the Router selected. Every step is logged to a reasoning trace table
at the end of the notebook.


## 1. Install Dependencies and Configure API Keys

In [1]:
# ============================================================
# 1. Install CrewAI and retrieval dependencies
# ============================================================
%pip install -qU crewai crewai-tools langchain-community tavily-python python-dotenv pandas


# ============================================================
# 2. Imports and Configuration
# ============================================================
import os
from pathlib import Path
from dotenv import load_dotenv

from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import PDFSearchTool

from langchain_community.tools.tavily_search import TavilySearchResults

# Disable CrewAI tracing messages
os.environ["CREWAI_TRACING_ENABLED"] = "false"

# Load only this project's own .env (not one elsewhere in the repo)
load_dotenv(Path.cwd() / ".env")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY was not found. Add it to your .env file."
    )
if not TAVILY_API_KEY:
    raise ValueError(
        "TAVILY_API_KEY was not found. Add it to your .env file "
        "(free key available at https://tavily.com)."
    )

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

PDF_PATH = os.path.join("data", "attention_is_all_you_need.pdf")

print("Configuration loaded.")
print(f"PDF source: {PDF_PATH}")


Note: you may need to restart the kernel to use updated packages.
Configuration loaded.
PDF source: data/attention_is_all_you_need.pdf


/var/folders/h0/6d0w9fr177z02gt0_sph25w40000gn/T/ipykernel_5700/3004679183.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


## 2. OpenAI LLM for CrewAI

In [2]:
# ============================================================
# OpenAI LLM for CrewAI
# ============================================================
llm = LLM(
    model="openai/gpt-4o-mini"
)


## 3. Define Tools

- `PDFSearchTool` (from `crewai_tools`) performs semantic search restricted
  to the uploaded PDF -- it handles its own chunking and embeddings
  internally, so no manual `PyPDFLoader` / text-splitter / FAISS setup is
  needed.
- `TavilySearchResults` performs a live web search.

Each tool is bound only to the agent that is allowed to use it, per
question -- see Step 5.


In [3]:
# ============================================================
# PDF Search Tool (crewai_tools)
# ============================================================
pdf_search_tool = PDFSearchTool(pdf=PDF_PATH)

# ============================================================
# Web Search Tool
# ============================================================
# NOTE: TavilySearchResults (from langchain_community) is a LangChain tool
# object, not a crewai.tools.BaseTool. Passing it directly into
# Agent(tools=[...]) raises:
#   pydantic ValidationError: Input should be a valid dictionary or
#   instance of BaseTool [type=model_type, input_type=TavilySearchResults]
# CrewAI validates every entry in `tools` against its own BaseTool class,
# so a LangChain tool has to be wrapped in a small CrewAI-native tool
# whose `_run` method delegates to the LangChain tool underneath.
from crewai.tools import BaseTool

_tavily = TavilySearchResults(max_results=5)

class TavilyWebSearchTool(BaseTool):
    name: str = "Web Search"
    description: str = (
        "Search the live web for current, real-world information using "
        "Tavily. Input should be a search query string."
    )

    def _run(self, query: str) -> str:
        results = _tavily.invoke(query)
        return str(results)

web_search_tool = TavilyWebSearchTool()

print("PDF tool ready ->", PDF_PATH)
print("Web tool ready  -> Tavily (wrapped as a CrewAI BaseTool, max_results=5)")


PDF tool ready -> data/attention_is_all_you_need.pdf
Web tool ready  -> Tavily (wrapped as a CrewAI BaseTool, max_results=5)


/var/folders/h0/6d0w9fr177z02gt0_sph25w40000gn/T/ipykernel_5700/214878314.py:19: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  _tavily = TavilySearchResults(max_results=5)


## 4. Create Agents: Router, Retriever, and Answer Generation

In [4]:
# ============================================================
# Router Agent
# ============================================================
router_agent = Agent(
    role="Router Agent",
    goal=(
        "Classify the user question and decide whether it needs PDF "
        "retrieval, web search, or can be answered directly."
    ),
    backstory=(
        "You are an intelligent router that decides the best retrieval "
        "path for user questions. You have two specialized sources "
        "available: a PDF vector search over the 'Attention Is All You "
        "Need' research paper, and a live web search for current or "
        "real-world information. If neither source is needed because the "
        "question is general knowledge, you route directly to an answer."
    ),
    llm=llm,
    verbose=True
)

# ============================================================
# Retriever Agent (built per question -- see Step 6)
# ============================================================
def make_retriever_agent(route):
    if route == "PDF_VECTOR_SEARCH":
        tools = [pdf_search_tool]
        backstory = (
            "You are a retrieval specialist who answers strictly using the "
            "uploaded 'Attention Is All You Need' paper via the PDF search "
            "tool. You always call the tool before answering and ground "
            "your evidence in what it returns."
        )
    elif route == "WEB_SEARCH":
        tools = [web_search_tool]
        backstory = (
            "You are a retrieval specialist who answers using live web "
            "search results. You always call the tool before answering "
            "and ground your evidence in what it returns."
        )
    else:  # DIRECT_ANSWER
        tools = []
        backstory = (
            "You are a knowledgeable retrieval specialist. For this "
            "question no retrieval tool is available or needed -- prepare "
            "the relevant evidence directly from your own knowledge."
        )

    return Agent(
        role="Retriever Agent",
        goal="Use the assigned tool (if any) to gather accurate evidence for answering the user question.",
        backstory=backstory,
        tools=tools,
        llm=llm,
        verbose=True
    )

# ============================================================
# Answer Generation Agent
# ============================================================
answer_agent = Agent(
    role="Answer Generation Agent",
    goal="Generate a clear, accurate, source-grounded answer using only the Retriever Agent's evidence.",
    backstory="You are an expert assistant who gives simple, accurate, evidence-based answers.",
    llm=llm,
    verbose=True
)

# Note: the project brief describes a single Retriever Agent that both
# executes retrieval and formulates the answer. This notebook splits that
# responsibility into two collaborating agents (Retriever Agent for
# tool-based evidence gathering, Answer Generation Agent for the final
# grounded response) to keep each agent's output separately inspectable in
# the reasoning trace log below. This still satisfies the brief's minimum
# of "at least two agents (router and retriever)".


## 5. Router Classification Step

Jupyter's kernel already runs its own asyncio event loop, and this version
of CrewAI detects that and refuses to run `crew.kickoff()` (its synchronous
entry point) from inside it -- it raises a `RuntimeError` instructing you to
use the async entry point instead. So every Crew in this notebook is run
with `await crew.kickoff_async()`, and every function that calls a Crew is
declared `async def`. Jupyter cells support top-level `await` natively, so
this doesn't require any extra setup.

The Router Agent reasons about each question and returns one of three exact
labels on its first line -- `PDF_VECTOR_SEARCH`, `WEB_SEARCH`, or
`DIRECT_ANSWER` -- which the code below parses to decide what happens next.


In [5]:
# ============================================================
# Classify Question (Router Crew)
# ============================================================
# NOTE: this function was originally written as a plain synchronous
# function using `def classify_question(question):` and `result =
# crew.kickoff()`. That version worked when run as a standalone script, but
# raised `RuntimeError: Agent execution was invoked synchronously from
# within a running event loop` when run inside a Jupyter notebook -- the
# Jupyter kernel already runs its own asyncio event loop in the background,
# and this version of CrewAI detects that and refuses to run its
# synchronous `crew.kickoff()` from inside it, rather than trying to work
# around it. The fix is to follow CrewAI's own suggestion in that error
# message and use its async entry point instead: the function is declared
# `async def`, and `crew.kickoff()` became `await crew.kickoff_async()`.
async def classify_question(question):
    task_router = Task(
        description=f"""
        Classify the following user question:

        Question:
        {question}

        Decide the retrieval path. Choose exactly one of these labels and
        write it as the first line of your answer:

        - PDF_VECTOR_SEARCH: if the question concerns the content of the
          uploaded research paper (the Transformer architecture, attention
          mechanism, training setup, results, authors, etc.)
        - WEB_SEARCH: if the question needs current, real-world, or
          time-sensitive information not contained in a 2017 research paper
        - DIRECT_ANSWER: if the question is general knowledge or reasoning
          that needs no retrieval at all

        Return:
        - Retrieval path (exact label on the first line)
        - Short reason
        """,
        agent=router_agent,
        expected_output="Retrieval path label followed by a short reason."
    )

    crew = Crew(
        agents=[router_agent],
        tasks=[task_router],
        process=Process.sequential,
        verbose=True
    )

    # Was: result = crew.kickoff()
    result = await crew.kickoff_async()
    result_text = str(result).upper()

    if "WEB_SEARCH" in result_text:
        route = "WEB_SEARCH"
    elif "PDF_VECTOR_SEARCH" in result_text:
        route = "PDF_VECTOR_SEARCH"
    else:
        route = "DIRECT_ANSWER"

    return route, str(result)


## 6. Agentic RAG Workflow (Retriever + Answer Generation)

In [6]:
# ============================================================
# Reasoning Trace Log
# ============================================================
trace_log = []

# ============================================================
# Agentic RAG Workflow
# ============================================================
# NOTE: this function was originally `def run_agentic_rag(question):`,
# calling `classify_question(question)` and `crew.kickoff()` directly
# (both synchronous). It hit the same Jupyter event-loop conflict described
# above the moment those calls tried to run inside a notebook cell. It's
# now `async def`, awaits `classify_question(...)` (itself async, see
# above), and awaits `crew.kickoff_async()` instead of calling
# `crew.kickoff()`. Because it's async, callers must now `await
# run_agentic_rag(...)` too -- see the demo loop below, which uses
# Jupyter's native support for top-level `await` in a cell.
async def run_agentic_rag(question):
    if not question.strip():
        return "Please enter a question."

    # --- Step 1: Router decides the retrieval path ------------------------
    route, router_reasoning = await classify_question(question)

    tool_used = {
        "PDF_VECTOR_SEARCH": "PDFSearchTool (crewai_tools)",
        "WEB_SEARCH": "TavilySearchResults",
        "DIRECT_ANSWER": "None (direct answer)",
    }[route]

    # --- Step 2: Retriever Agent is built with only the matching tool ------
    retriever_agent = make_retriever_agent(route)

    if route == "PDF_VECTOR_SEARCH":
        retrieval_instruction = (
            "Use the PDF search tool to find information relevant to the "
            "question below, then summarize the relevant evidence you found."
        )
    elif route == "WEB_SEARCH":
        retrieval_instruction = (
            "Use the web search tool to find information relevant to the "
            "question below, then summarize the relevant evidence you found."
        )
    else:
        retrieval_instruction = (
            "No retrieval tool is available for this question. Prepare the "
            "relevant evidence directly from your own knowledge."
        )

    task_retriever = Task(
        description=f"""
        {retrieval_instruction}

        User Question:
        {question}
        """,
        agent=retriever_agent,
        expected_output="Relevant evidence gathered for the user's question, with its source noted (page number or URL) when available."
    )

    # --- Step 3: Answer Generation Agent writes the final answer ------------
    task_answer = Task(
        description=f"""
        Create the final answer for the user using the Retriever Agent's
        evidence.

        User Question:
        {question}

        Rules:
        - Use only the Retriever Agent's evidence above (if any was gathered).
        - Do not make up information.
        - Explain in a simple and clear way.
        - Mention page numbers or source URLs where available.
        - If the answer is not available in the evidence, say so explicitly.
        """,
        agent=answer_agent,
        expected_output="Final source-grounded answer.",
        context=[task_retriever]
    )

    crew = Crew(
        agents=[retriever_agent, answer_agent],
        tasks=[task_retriever, task_answer],
        process=Process.sequential,
        verbose=True
    )

    # Was: result = crew.kickoff()
    result = await crew.kickoff_async()
    # answer, so the trace log shows each agent's distinct contribution.
    retriever_evidence = str(result.tasks_output[0].raw)
    final_answer = str(result.tasks_output[-1].raw)

    trace_log.append({
        "question": question,
        "route": route,
        "router_reasoning": router_reasoning,
        "tool_used": tool_used,
        "retriever_agent_evidence": retriever_evidence,
        "answer_agent_output": final_answer,
    })

    return final_answer


## 7. Demo Run

Five sample questions covering all three routes: two grounded in the PDF,
one requiring live web search, one general-knowledge question needing no
retrieval, and one edge case to show the system admitting it can't find an
answer rather than hallucinating.


In [7]:
demo_questions = [
    "What is multi-head attention and why does the Transformer use it instead of a single attention head?",
    "What BLEU score did the Transformer (big) model achieve on the WMT 2014 English-to-French translation task?",
    "What is the latest large language model OpenAI has released?",
    "What's 15% of 240?",
    "What's the capital of France?",
    "What is the Transformer paper's stance on quantum computing?",  # not covered -> should not hallucinate
]

for q in demo_questions:
    print(f"\n{'=' * 70}\nQUESTION: {q}\n{'=' * 70}")
    # Was: final_answer = run_agentic_rag(q)
    # run_agentic_rag is now async (see Step 6), so its result must be
    # awaited. Jupyter cells support top-level `await` natively, so this
    # loop doesn't need to be wrapped in its own async function.
    final_answer = await run_agentic_rag(q)
    print(f"\nFINAL ANSWER:\n{final_answer}")



QUESTION: What is multi-head attention and why does the Transformer use it instead of a single attention head?


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: bf2425c7-9cac-48fb-a297-d7617ec769a1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│          Classify the following user question:                                                                  │
│                                                                                                                 │
│          Question:                                                                                              │
│          What is multi-head attention and why does the Transformer use it instead of a single attention head?   │
│                                                                                                                 │
│          Decide the retrieval path. Choose exactly one of these labels and                                      │
│          write it as the first line of your answer:                                                             │
│                                                                                                                 │
│          - PDF_VECTOR_SEARCH: if the question concerns the content of the                                       │
│            uploaded research paper (the Transformer architecture, attention                                     │
│            mechanism, training setup, results, authors, etc.)                                                   │
│          - WEB_SEARCH: if the question needs current, real-world, or                                            │
│            time-sensitive information not contained in a 2017 research paper                                    │
│          - DIRECT_ANSWER: if the question is general knowledge or reasoning                                     │
│            that needs no retrieval at all                                                                       │
│                                                                                                                 │
│          Return:                                                                                                │
│          - Retrieval path (exact label on the first line)                                                       │
│          - Short reason                                                                                         │
│                                                                                                                 │
│  ID: c96cd9d9-aeda-44ff-a19e-c1cb09c11abd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Router Agent                                                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│          Classify the following user question:                                                                  │
│                                                                                                                 │
│          Question:                                                                                              │
│          What is multi-head attention and why does the Transformer use it instead of a single attention head?   │
│                                                                                                                 │
│          Decide the retrieval path. Choose exactly one of these labels and                                      │
│          write it as the first line of your answer:                                                             │
│                                                                                                                 │
│          - PDF_VECTOR_SEARCH: if the question concerns the content of the                                       │
│            uploaded research paper (the Transformer architecture, attention                                     │
│            mechanism, training setup, results, authors, etc.)                                                   │
│          - WEB_SEARCH: if the question needs current, real-world, or                                            │
│            time-sensitive information not contained in a 2017 research paper                                    │
│          - DIRECT_ANSWER: if the question is general knowledge or reasoning                                     │
│            that needs no retrieval at all                                                                       │
│                                                                                                                 │
│          Return:                                                                                                │
│          - Retrieval path (exact label on the first line)                                                       │
│          - Short reason                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Answer Generation Agent                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The Transformer paper does not explicitly mention quantum computing. The document primarily focuses on         │
│  improvements and innovations in neural network architectures, particularly the introduction of the             │
│  Transformer model that relies entirely on an attention mechanism rather than recurrence. There is no           │
│  reference or stance regarding quantum computing within the paper's content based on the search results.        │
│                                                                                                                 │
│  For further reference, this information can be found throughout the paper, particularly in the introductory    │
│  sections where the authors discuss the limitations of recurrent models and their approach in proposing the     │
│  Transformer architecture.                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│          Create the final answer for the user using the Retriever Agent's                                       │
│          evidence.                                                                                              │
│                                                                                                                 │
│          User Question:                                                                                         │
│          What is the Transformer paper's stance on quantum computing?                                           │
│                                                                                                                 │
│          Rules:                                                                                                 │
│          - Use only the Retriever Agent's evidence above (if any was gathered).                                 │
│          - Do not make up information.                                                                          │
│          - Explain in a simple and clear way.                                                                   │
│          - Mention page numbers or source URLs where available.                                                 │
│          - If the answer is not available in the evidence, say so explicitly.                                   │
│                                                                                                                 │
│  Agent: Answer Generation Agent                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 912030ac-e35f-4927-9d6c-f6f764a1ca82                                                                       │
│  Final Output: The Transformer paper does not explicitly mention quantum computing. The document primarily      │
│  focuses on improvements and innovations in neural network architectures, particularly the introduction of the  │
│  Transformer model that relies entirely on an attention mechanism rather than recurrence. There is no           │
│  reference or stance regarding quantum computing within the paper's content based on the search results.        │
│                                                                                                                 │
│  For further reference, this information can be found throughout the paper, particularly in the introductory    │
│  sections where the authors discuss the limitations of recurrent models and their approach in proposing the     │
│  Transformer architecture.                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL ANSWER:
The Transformer paper does not explicitly mention quantum computing. The document primarily focuses on improvements and innovations in neural network architectures, particularly the introduction of the Transformer model that relies entirely on an attention mechanism rather than recurrence. There is no reference or stance regarding quantum computing within the paper's content based on the search results. 

For further reference, this information can be found throughout the paper, particularly in the introductory sections where the authors discuss the limitations of recurrent models and their approach in proposing the Transformer architecture.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 8. Reasoning Trace Table

In [8]:
import pandas as pd

trace_df = pd.DataFrame(trace_log)
pd.set_option("display.max_colwidth", 120)
trace_df


,question,route,router_reasoning,tool_used,retriever_agent_evidence,answer_agent_output
0,What is multi-head attention and why does the Transformer use it instead of a single attention head?,PDF_VECTOR_SEARCH,"PDF_VECTOR_SEARCH: The question pertains to the specifics of the Transformer architecture, particularly the multi-he...",PDFSearchTool (crewai_tools),The information regarding multi-head attention and the reasons it is used in the Transformer model can be found in t...,Multi-head attention is a mechanism used in the Transformer model that allows it to attend to different parts of the...
1,What BLEU score did the Transformer (big) model achieve on the WMT 2014 English-to-French translation task?,PDF_VECTOR_SEARCH,- PDF_VECTOR_SEARCH: The question specifically asks for the BLEU score achieved by the Transformer (big) model on a ...,PDFSearchTool (crewai_tools),The Transformer (big) model achieved a BLEU score of **41.8** on the WMT 2014 English-to-French translation task. Th...,The Transformer (big) model achieved a BLEU score of **41.8** on the WMT 2014 English-to-French translation task. Th...
2,What is the latest large language model OpenAI has released?,WEB_SEARCH,"WEB_SEARCH: This question requires current, real-world information about the latest developments from OpenAI, which ...",TavilySearchResults,"The latest large language model released by OpenAI is **GPT-4o**, which was released in May 2024. This model represe...","The latest large language model released by OpenAI is **GPT-4o**, which was released in May 2024. This model represe..."
3,What's 15% of 240?,DIRECT_ANSWER,DIRECT_ANSWER: This is a simple mathematical calculation that requires no retrieval. \n\n15% of 240 is calculated by...,None (direct answer),"To calculate 15% of 240, you can follow these steps:\n\n1. Convert the percentage to a decimal: 15% = 0.15.\n2. Mult...","To calculate 15% of 240, you can follow these steps:\n\n1. Convert the percentage to a decimal: 15% = 0.15.\n2. Mult..."
4,What's the capital of France?,DIRECT_ANSWER,"DIRECT_ANSWER: This question pertains to general knowledge, and the capital of France is a well-known fact (Paris).",None (direct answer),"The capital of France is Paris. This information is well-documented in geography and political resources, and Paris ...",The capital of France is Paris. This information is well-documented in geography and political resources. Paris is k...
5,What is the Transformer paper's stance on quantum computing?,PDF_VECTOR_SEARCH,"PDF_VECTOR_SEARCH: The question specifically asks about the stance of the Transformer paper on quantum computing, wh...",PDFSearchTool (crewai_tools),The Transformer paper does not explicitly mention quantum computing. The document primarily focuses on improvements ...,The Transformer paper does not explicitly mention quantum computing. The document primarily focuses on improvements ...


In [9]:
os.makedirs("outputs", exist_ok=True)
trace_df.to_csv("outputs/reasoning_trace_log.csv", index=False)
print("Saved trace log to outputs/reasoning_trace_log.csv")


Saved trace log to outputs/reasoning_trace_log.csv


## 9. Observations

Run the cells above with valid API keys, then note here:
- Did the Router correctly classify all five demo questions?
- Did the "not covered in the paper" question correctly avoid hallucinating?
- Any misroutes you observed, and what in the question likely caused them?

(See `README.md` for the full architecture write-up, agent roles,
coordination flow, and the challenges/trade-offs discussion required for
submission.)
